# Victor — Demo + Fine-Tune DeBERTa on SciFact

**What this notebook does:**
1. Shows what Victor has built (demo)
2. Fine-tunes DeBERTa-v3-base on SciFact (GPU)
3. Benchmarks the fine-tuned model
4. Answers common questions

**Runtime:** T4 GPU recommended. ~15-20 min total.

## Common Questions (Answered First)

**Q: Why fine-tune DeBERTa instead of using a bigger LLM?**
A: DeBERTa-v3-base (86M params) fine-tuned on SciFact hit 88% F1 (SciClaims paper). Qwen2.5-7B alone only gets 59%. The NLI cross-encoder is the right tool for this job — it was trained specifically for entailment/contradiction.

**Q: Won't fine-tuning on SciFact cause contamination?**
A: Only if you test on the same split you trained on. We train on `claims_train.jsonl` (809 claims) and test on `claims_dev.jsonl` (300 claims). These are separate splits. Do NOT use pre-fine-tuned models from HuggingFace unless their card confirms dev-set exclusion.

**Q: Why does DeBERTa fail on long text in primary mode?**
A: DeBERTa-v3-base was fine-tuned on MNLI (short sentence pairs, ~10-30 words each). SciFact pairs a ~20-word claim with a ~200-word abstract. The model defaults to "neutral" because the inputs are outside its training distribution.

**Q: Will fine-tuning fix the long-text problem?**
A: Yes — fine-tuning on SciFact pairs (which ARE long) teaches the model to handle this exact input length. SciClaims proved this works (88% F1).

**Q: What if I don't have GPU?**
A: CPU fine-tuning works but is ~10x slower (~2-3 hours vs ~15 min). Use Google Colab free tier for GPU.

## Step 1: Install Dependencies

In [ ]:
# Install Victor and dependencies
!pip install -q sentence-transformers chromadb networkx click rich pydantic

# Clone Victor repo (or upload your code)
!git clone https://github.com/YOUR_USERNAME/victor.git 2>/dev/null || echo 'Using uploaded code'
%cd victor
!pip install -e . -q

print('✓ Dependencies installed')

## Step 2: Upload Your Code (if not cloning)

If you're not cloning from GitHub, upload your `victor_v5_3_complete_2026-08-12.zip` file:

In [ ]:
# Option A: Upload zip file
# from google.colab import files
# uploaded = files.upload()  # Select your zip file
# !unzip -q victor_v5_3_complete_2026-08-12.zip
# %cd victor_fixed
# !pip install -e . -q

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/victor
# !pip install -e . -q

print('✓ Code ready')

## Step 3: Demo — See What's Built

In [ ]:
import sys
sys.path.insert(0, '.')

from victor import Victor
from victor.config import VictorConfig
from pathlib import Path

# Initialize with mock models (fast, no download needed)
config = VictorConfig(
    use_mock_embedder=True,
    use_mock_nli=True,
    use_mock_llm=True,
)
v = Victor(config)

print('✓ Victor initialized')
print(f'  Nodes: {v.graph._g.number_of_nodes()}')
print(f'  Edges: {v.graph._g.number_of_edges()}')
print(f'  Retriever: {type(v.retriever).__name__}')
print(f'  Tools: {len(v.tool_registry.list_tools())}')

In [ ]:
# Ingest a sample paper
sample_paper = Path('examples/sample_paper.md')
if sample_paper.exists():
    result = v.ingest_file(sample_paper, source_type='paper')
    print(f'✓ Ingested: {result.get("title", "Unknown")}')
    print(f'  Chunks: {result.get("n_chunks", 0)}')
    print(f'  Claims: {result.get("n_claims", 0)}')
    print(f'  Concepts: {result.get("n_concepts", 0)}')
else:
    print('⚠ Sample paper not found')

In [ ]:
# Show knowledge graph stats
stats = v.stats()
print('Knowledge Graph Stats:')
print(f'  Nodes: {stats["graph"]["nodes"]}')
print(f'  Edges: {stats["graph"]["edges"]}')
print(f'  Retriever: {stats["retriever_type"]}')
print(f'  Tools: {stats["tool_count"]}')

In [ ]:
# Demo masked memory — build claim nodes directly so this works
# even under mock mode (mock LLM correctly extracts zero claims,
# so v.list_claims() would be empty here otherwise).
from victor.models import EdgeType

secret_text = "Unpublished: compound X shows 40% efficacy — embargoed until peer review"
public_text = "Published: compound Y shows mild neuroprotection at high doses"

v.graph.add_node("demo_paper", "paper", "Demo paper", {})
v.graph.add_node("demo_claim_secret", "claim", secret_text, {"epistemic_status": "literature"})
v.graph.add_node("demo_claim_public", "claim", public_text, {"epistemic_status": "literature"})
v.graph.add_edge("demo_paper", "demo_claim_secret", EdgeType.MAKES_CLAIM)
v.graph.add_edge("demo_paper", "demo_claim_public", EdgeType.MAKES_CLAIM)

print("BEFORE masking:")
print(" ", v.graph.get_node("demo_claim_secret")["label"])

v.mask("demo_claim_secret")

print()
print("AFTER masking — calling find_gaps() (the function that must never leak):")
from victor.reasoning import EvidenceWeighter, GapDetector
weighter = EvidenceWeighter(v.graph)
gaps = GapDetector(v.graph, weighter).find_gaps()
for g in gaps:
    if g.claim_id in ("demo_claim_secret", "demo_claim_public"):
        print(" ", f"[{g.claim_id}]", g.claim_text)

v.unmask("demo_claim_secret")
print()
print("AFTER unmasking — content restored:")
print(" ", v.graph.get_node("demo_claim_secret")["label"])

## Step 4: Prepare SciFact Data for Fine-Tuning

Upload your SciFact data files, or use Victor's built-in loader:

In [ ]:
# Upload SciFact data (from your local data/ folder)
# from google.colab import files
# uploaded = files.upload()  # Upload claims_train.jsonl, claims_dev.jsonl, corpus.jsonl

# Or use Victor's built-in loader (already correct for this schema)
from victor.scifact_benchmark import load_scifact_data

train_samples = load_scifact_data(
    corpus_path="data/corpus.jsonl",
    claims_path="data/claims_train.jsonl",
)
eval_samples = load_scifact_data(
    corpus_path="data/corpus.jsonl",
    claims_path="data/claims_dev.jsonl",
)

print(f'✓ Train samples: {len(train_samples)}')
print(f'✓ Eval samples: {len(eval_samples)}')

## Step 5: Fine-Tune DeBERTa-v3-base on SciFact

**What this does:**
- Takes DeBERTa-v3-base (86M params)
- Fine-tunes on SciFact training set (809 claims)
- Evaluates on dev set (300 claims)
- Saves the fine-tuned model

**Expected result:** ~85-88% F1 (based on SciClaims paper)

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np

print(f'✓ PyTorch: {torch.__version__}')
print(f'✓ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load tokenizer and model
model_name = 'cross-encoder/nli-deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f'✓ Model loaded: {model_name}')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Prepare dataset for fine-tuning
# load_scifact_data() returns EvalSample objects with:
#   .claim_text, .abstract_text, .gold_direction
#   (one of "confirm"/"contradict"/"inconclusive")
#
# This model's own label order (verified via the label-order smoke test
# in reasoning.py) is: 0=contradiction, 1=neutral, 2=entailment.
# Map Victor's direction strings to THAT order.
DIRECTION_TO_LABEL = {"contradict": 0, "inconclusive": 1, "confirm": 2}

class SciFactDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        encoding = self.tokenizer(
            s.claim_text,
            s.abstract_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(DIRECTION_TO_LABEL[s.gold_direction], dtype=torch.long),
        }

train_dataset = SciFactDataset(train_samples, tokenizer)
eval_dataset = SciFactDataset(eval_samples, tokenizer)

print(f'✓ Train dataset: {len(train_dataset)} samples')
print(f'✓ Eval dataset: {len(eval_dataset)} samples')

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
)

# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    from sklearn.metrics import f1_score
    
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_per_class = f1_score(labels, predictions, average=None)
    
    return {
        'f1': f1_macro,
        'f1_contradict': f1_per_class[0],
        'f1_inconclusive': f1_per_class[1],
        'f1_confirm': f1_per_class[2],
    }

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print('✓ Trainer initialized')

In [ ]:
# Train the model
print('Starting fine-tuning...')
print('This will take ~15-20 min on T4 GPU, ~2-3 hours on CPU.')
print()

trainer.train()

print('\n✓ Fine-tuning complete!')

In [ ]:
# Evaluate the fine-tuned model
results = trainer.evaluate()

print('Fine-Tuned Model Results:')
print(f'  F1 (macro): {results["eval_f1"]:.3f}')
print(f'  F1 (contradict): {results["eval_f1_contradict"]:.3f}')
print(f'  F1 (inconclusive): {results["eval_f1_inconclusive"]:.3f}')
print(f'  F1 (confirm): {results["eval_f1_confirm"]:.3f}')
print()
print('Comparison:')
print(f'  Victor cross-check (zero-shot): 71.0%')
print(f'  SciClaims (fine-tuned, published): 88.0%')
print(f'  This model: {results["eval_f1"]*100:.1f}%')

In [ ]:
# Save the fine-tuned model
model_path = './deberta-v3-base-scifact-finetuned'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f'✓ Model saved to: {model_path}')
print(f'  Files: {list(Path(model_path).glob("*"))}')

## Step 6: Test the Fine-Tuned Model with Victor

Now use your fine-tuned model as the NLI cross-encoder in Victor:

In [ ]:
from victor.reasoning import NLICrossEncoder

# Load the fine-tuned model
nli = NLICrossEncoder(model_name=model_path)

# Test on known pairs
test_pairs = [
    ('Drug X reduces blood pressure.', 'In a trial, Drug X lowered systolic BP by 12 mmHg.', 'confirm'),
    ('Drug X reduces blood pressure.', 'Drug X produced no significant change in BP.', 'contradict'),
    ('Drug X reduces blood pressure.', 'Drug X was well-tolerated with mild side effects.', 'inconclusive'),
]

print('Fine-Tuned Model Tests:')
for premise, hypothesis, expected in test_pairs:
    direction, conf = nli.predict(premise, hypothesis)
    status = '✓' if direction == expected else '✗'
    print(f'  {status} {premise[:40]}... -> {direction} (conf={conf:.2f}), expected {expected}')

In [ ]:
# Run SciFact benchmark with fine-tuned model
# This uses Victor's built-in benchmark harness

import subprocess
import os

# Set Ollama host (needed for LLM in cross-check mode)
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'

# Run benchmark
cmd = [
    'python', '-m', 'victor.scifact_benchmark',
    '--corpus-path', 'data/corpus.jsonl',
    '--claims-path', 'data/claims_dev.jsonl',
    '--n-samples', '100',
    '--nli-model', model_path,
    '--nli-primary',  # Use fine-tuned NLI as primary judge
]

print('Running benchmark with fine-tuned model...')
print(' '.join(cmd))
print()

# Uncomment to run:
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
# if result.stderr:
#     print('Errors:', result.stderr)

## Step 7: Download Your Fine-Tuned Model

Download the fine-tuned model to use locally with Victor:

In [ ]:
# Zip the model for download
!zip -r deberta-v3-base-scifact-finetuned.zip {model_path}

# Download
from google.colab import files
files.download('deberta-v3-base-scifact-finetuned.zip')

print('✓ Model downloaded!')
print('  Extract to your Victor directory')
print('  Run: python -m victor.scifact_benchmark --nli-model ./deberta-v3-base-scifact-finetuned --nli-primary')

## Summary

**What you just did:**
1. ✓ Saw what Victor has built (demo)
2. ✓ Fine-tuned DeBERTa-v3-base on SciFact
3. ✓ Evaluated the fine-tuned model
4. ✓ Saved and downloaded the model

**Next steps locally:**
1. Extract the downloaded model to your Victor directory
2. Run the benchmark:
   ```bash
   OLLAMA_HOST=http://127.0.0.1:11434 python -m victor.scifact_benchmark \
     --corpus-path data/corpus.jsonl \
     --claims-path data/claims_dev.jsonl \
     --n-samples 100 \
     --nli-model ./deberta-v3-base-scifact-finetuned \
     --nli-primary
   ```
3. If accuracy > 85%, DoD #4 is closed!

**Key numbers:**
- Victor cross-check (zero-shot): 71.0%
- SciClaims (fine-tuned): 88.0%
- Your target: 85%+